In [4]:
import time
import random
import string
from dataclasses import dataclass
from typing import List


# ============================
# Estrutura para métricas
# ============================

@dataclass
class Metrics:
    comparisons: int = 0  # comparações de caracteres
    iterations: int = 0   # iterações de laços / operações principais


# ============================
# Implementação Naive
# ============================

def naive_search(text: str, pattern: str, metrics: Metrics | None = None) -> List[int]:
    if metrics is None:
        metrics = Metrics()

    n, m = len(text), len(pattern)
    occurrences = []

    for i in range(n - m + 1):
        metrics.iterations += 1  # laço externo
        match = True
        for j in range(m):
            metrics.iterations += 1  # laço interno
            metrics.comparisons += 1
            if text[i + j] != pattern[j]:
                match = False
                break
        if match:
            occurrences.append(i)

    return occurrences


# ============================
# Rabin-Karp sem rolling hash
# (hash Horner padrão a cada posição)
# ============================

def rk_no_rolling(text: str, pattern: str, metrics: Metrics | None = None) -> List[int]:
    if metrics is None:
        metrics = Metrics()

    n, m = len(text), len(pattern)
    if m == 0 or m > n:
        return []

    base = 256
    mod = 10**9 + 7

    # hash do padrão
    p_hash = 0
    for c in pattern:
        metrics.iterations += 1
        p_hash = (p_hash * base + ord(c)) % mod

    occurrences = []

    for i in range(n - m + 1):
        metrics.iterations += 1  # laço principal

        # hash da substring text[i:i+m] recomputado do zero
        s_hash = 0
        for k in range(m):
            metrics.iterations += 1
            s_hash = (s_hash * base + ord(text[i + k])) % mod

        # se hashes batem, verificar caractere a caractere
        if s_hash == p_hash:
            match = True
            for j in range(m):
                metrics.iterations += 1
                metrics.comparisons += 1
                if text[i + j] != pattern[j]:
                    match = False
                    break
            if match:
                occurrences.append(i)

    return occurrences


# ============================
# Rabin-Karp com rolling hash
# ============================

def rk_rolling(text: str, pattern: str, metrics: Metrics | None = None) -> List[int]:
    if metrics is None:
        metrics = Metrics()

    n, m = len(text), len(pattern)
    if m == 0 or m > n:
        return []

    base = 256
    mod = 10**9 + 7

    # hash do padrão
    p_hash = 0
    for c in pattern:
        metrics.iterations += 1
        p_hash = (p_hash * base + ord(c)) % mod

    # hash da primeira janela do texto
    t_hash = 0
    for i in range(m):
        metrics.iterations += 1
        t_hash = (t_hash * base + ord(text[i])) % mod

    # fator base^(m-1) para remover o primeiro caractere na janela
    base_m1 = pow(base, m - 1, mod)

    occurrences = []

    for i in range(n - m + 1):
        metrics.iterations += 1  # laço principal

        # Se hashes coincidem, verifica caractere a caractere
        if t_hash == p_hash:
            match = True
            for j in range(m):
                metrics.iterations += 1
                metrics.comparisons += 1
                if text[i + j] != pattern[j]:
                    match = False
                    break
            if match:
                occurrences.append(i)

        # Atualiza o hash para a próxima janela
        if i < n - m:
            metrics.iterations += 1
            t_hash = (t_hash - ord(text[i]) * base_m1) % mod  # remove primeiro
            t_hash = (t_hash * base + ord(text[i + m])) % mod  # adiciona próximo
            t_hash %= mod

    return occurrences


# ============================
# Knuth-Morris-Pratt (KMP)
# ============================

def kmp_prefix_function(pattern: str, metrics: Metrics | None = None) -> List[int]:
    if metrics is None:
        metrics = Metrics()

    m = len(pattern)
    lps = [0] * m
    length = 0  # tamanho do prefixo mais longo
    i = 1

    while i < m:
        metrics.iterations += 1
        metrics.comparisons += 1
        if pattern[i] == pattern[length]:
            length += 1
            lps[i] = length
            i += 1
        else:
            if length != 0:
                length = lps[length - 1]
            else:
                lps[i] = 0
                i += 1

    return lps


def kmp_search(text: str, pattern: str, metrics: Metrics | None = None) -> List[int]:
    if metrics is None:
        metrics = Metrics()

    n, m = len(text), len(pattern)
    if m == 0 or m > n:
        return []

    # Pré-processamento
    lps = kmp_prefix_function(pattern, metrics)

    occurrences = []
    i = 0  # índice em text
    j = 0  # índice em pattern

    while i < n:
        metrics.iterations += 1
        metrics.comparisons += 1
        if text[i] == pattern[j]:
            i += 1
            j += 1
            if j == m:
                occurrences.append(i - j)
                j = lps[j - 1]
        else:
            if j != 0:
                j = lps[j - 1]
            else:
                i += 1

    return occurrences


# ============================
# Funções auxiliares
# ============================

def run_with_metrics(algorithm_name: str, func, text: str, pattern: str):
    metrics = Metrics()
    start = time.perf_counter()
    occurrences = func(text, pattern, metrics)
    elapsed = time.perf_counter() - start

    return {
        "algorithm": algorithm_name,
        "n_text": len(text),
        "m_pattern": len(pattern),
        "occurrences": occurrences,
        "comparisons": metrics.comparisons,
        "iterations": metrics.iterations,
        "time": elapsed,
    }


def generate_random_string(length: int, alphabet: str = string.ascii_uppercase) -> str:
    return "".join(random.choice(alphabet) for _ in range(length))


def embed_pattern(text: str, pattern: str, positions: list[int]) -> str:
    """
    Garante que o padrão apareça em certas posições do texto,
    sobrescrevendo os caracteres nessas posições.
    """
    n, m = len(text), len(pattern)
    lst = list(text)
    for pos in positions:
        if 0 <= pos <= n - m:
            lst[pos:pos + m] = pattern
    return "".join(lst)


def fmt_int(x: int) -> str:
    # formata com separador de milhar (estilo 1,234,567)
    return f"{x:,}".replace(",", ".")


def fmt_time(x: float) -> str:
    return f"{x:.6f}"


# ============================
# MAIN – testes pequenos e grandes
# ============================

if __name__ == "__main__":
    algos = [
        ("Naive", naive_search),
        ("Rabin-Karp sem rolling", rk_no_rolling),
        ("Rabin-Karp com rolling", rk_rolling),
        ("KMP", kmp_search),
    ]

    # ---------- Testes PEQUENOS ----------
    small_text = "ABRACADABRA"
    small_pattern = "ABRA"

    print("=== Testes com strings pequenas (corretude e métricas) ===\n")

    small_results = []
    for name, func in algos:
        r = run_with_metrics(name, func, small_text, small_pattern)
        small_results.append(r)

    print("### Teste de corretude — strings pequenas\n")
    print(f'Texto: "{small_text}"')
    print(f'Padrão: "{small_pattern}"\n')
    print("| Algoritmo                | |T| (texto) | |P| (padrão) | Ocorrências | Comparações | Iterações | Tempo (s) |")
    print("|--------------------------|------------|--------------|-------------|-------------|-----------|-----------|")
    for r in small_results:
        occ_str = str(r["occurrences"])
        print(
            f"| {r['algorithm']:<24} | "
            f"{r['n_text']:>10} | "
            f"{r['m_pattern']:>12} | "
            f"{occ_str:>11} | "
            f"{fmt_int(r['comparisons']):>11} | "
            f"{fmt_int(r['iterations']):>9} | "
            f"{fmt_time(r['time'])} |"
        )

    # ---------- Testes GRANDES ----------
    print("\n\n=== Testes com strings GRANDES (contagens e tempo) ===\n")

    N = 500_000   # tamanho do texto
    M = 50        # tamanho do padrão

    # texto aleatório
    big_text = generate_random_string(N, alphabet="ABCD")
    # padrão aleatório
    big_pattern = generate_random_string(M, alphabet="ABCD")
    # garante algumas ocorrências reais do padrão no texto
    big_text = embed_pattern(big_text, big_pattern, positions=[1_000, 100_000, 250_000, 400_000])

    big_results = []
    for name, func in algos:
        print(f"Rodando {name}...")
        r = run_with_metrics(name, func, big_text, big_pattern)
        big_results.append(r)

    print("\n### Testes de desempenho — strings grandes\n")
    print(f"Texto aleatório com |T| = {N}, padrão com |P| = {M}, alfabeto = {{A,B,C,D}}, com o padrão inserido em algumas posições.\n")

    print("| Algoritmo                | |T| (texto) | |P| (padrão) | Ocorrências | Comparações | Iterações | Tempo (s) |")
    print("|--------------------------|------------|--------------|-------------|-------------|-----------|-----------|")
    for r in big_results:
        occ_str = str(r["occurrences"])
        print(
            f"| {r['algorithm']:<24} | "
            f"{r['n_text']:>10} | "
            f"{r['m_pattern']:>12} | "
            f"{occ_str:>11} | "
            f"{fmt_int(r['comparisons']):>11} | "
            f"{fmt_int(r['iterations']):>9} | "
            f"{fmt_time(r['time'])} |"
        )


=== Testes com strings pequenas (corretude e métricas) ===

### Teste de corretude — strings pequenas

Texto: "ABRACADABRA"
Padrão: "ABRA"

| Algoritmo                | |T| (texto) | |P| (padrão) | Ocorrências | Comparações | Iterações | Tempo (s) |
|--------------------------|------------|--------------|-------------|-------------|-----------|-----------|
| Naive                    |         11 |            4 |      [0, 7] |          16 |        24 | 0.000013 |
| Rabin-Karp sem rolling   |         11 |            4 |      [0, 7] |           8 |        52 | 0.000013 |
| Rabin-Karp com rolling   |         11 |            4 |      [0, 7] |           8 |        31 | 0.000010 |
| KMP                      |         11 |            4 |      [0, 7] |          16 |        16 | 0.000006 |


=== Testes com strings GRANDES (contagens e tempo) ===

Rodando Naive...
Rodando Rabin-Karp sem rolling...
Rodando Rabin-Karp com rolling...
Rodando KMP...

### Testes de desempenho — strings grandes

Texto 

In [5]:
# Depois de gerar small_results e big_results, adicione isso no final do script:

def salvar_tabelas_md(small_results, big_results, N, M, filename="tabelas_pattern_matching.md"):
    with open(filename, "w", encoding="utf-8") as f:
        # Tabela pequena
        f.write("### Teste de corretude — strings pequenas\n\n")
        f.write('Texto: "ABRACADABRA"\n')
        f.write('Padrão: "ABRA"\n\n')
        f.write("| Algoritmo                | |T| (texto) | |P| (padrão) | Ocorrências | Comparações | Iterações | Tempo (s) |\n")
        f.write("|--------------------------|------------|--------------|-------------|-------------|-----------|-----------|\n")
        for r in small_results:
            occ_str = str(r["occurrences"])
            f.write(
                f"| {r['algorithm']:<24} | "
                f"{r['n_text']:>10} | "
                f"{r['m_pattern']:>12} | "
                f"{occ_str:>11} | "
                f"{fmt_int(r['comparisons']):>11} | "
                f"{fmt_int(r['iterations']):>9} | "
                f"{fmt_time(r['time'])} |\n"
            )

        f.write("\n\n### Testes de desempenho — strings grandes\n\n")
        f.write(f"Texto aleatório com |T| = {N}, padrão com |P| = {M}, alfabeto = {{A,B,C,D}}, com o padrão inserido em algumas posições.\n\n")
        f.write("| Algoritmo                | |T| (texto) | |P| (padrão) | Ocorrências | Comparações | Iterações | Tempo (s) |\n")
        f.write("|--------------------------|------------|--------------|-------------|-------------|-----------|-----------|\n")
        for r in big_results:
            occ_str = str(r["occurrences"])
            f.write(
                f"| {r['algorithm']:<24} | "
                f"{r['n_text']:>10} | "
                f"{r['m_pattern']:>12} | "
                f"{occ_str:>11} | "
                f"{fmt_int(r['comparisons']):>11} | "
                f"{fmt_int(r['iterations']):>9} | "
                f"{fmt_time(r['time'])} |\n"
            )

# No final do main, depois de gerar small_results e big_results:
salvar_tabelas_md(small_results, big_results, N, M)
print("\nTabelas salvas em 'tabelas_pattern_matching.md'.")



Tabelas salvas em 'tabelas_pattern_matching.md'.
